<a href="https://colab.research.google.com/github/wojciechwargas-hash/Wojciech-Wargas/blob/main/Smart_Finance_Assistant_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SMART FINANCE ASSISTANT
# ISYS2001 - Introduction to Business Programming
# ============================================================

# ── STEP 1: INSTALL AND IMPORT ───────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'hands-on-ai', 'gradio', 'pandas', '--quiet'], check=True)

import os
import math
import tempfile
import pandas as pd

from hands_on_ai.chat  import get_response
from hands_on_ai.rag   import chunk_text, get_embeddings, save_index_with_sources, get_top_k
from hands_on_ai.agent import run_agent, register_tool

import gradio as gr

# ── STEP 2: CONFIGURATION ────────────────────────────────────
# Set the server URL and model
os.environ['HANDS_ON_AI_SERVER'] = 'http://localhost:11434'
os.environ['HANDS_ON_AI_MODEL']  = 'llama3'
# Increase timeout to 120 seconds to prevent 'Request timed out' errors
os.environ['HANDS_ON_AI_TIMEOUT'] = '120'

print('Packages imported and configuration set (Timeout increased to 120s)')

# ── STEP 3: CREATE DATA FILES ────────────────────────────────
os.makedirs('data', exist_ok=True)

csv_content = """date,category,description,amount,type
2025-01-03,Food,Coles grocery run,87.50,expense
2025-01-05,Transport,Fuel BP Midland,78.20,expense
2025-01-07,Food,McDonalds lunch,14.90,expense
2025-01-08,Entertainment,Netflix subscription,22.99,expense
2025-01-10,Health,Chemist Warehouse vitamins,34.50,expense
2025-01-12,Food,IGA grocery top-up,42.30,expense
2025-01-14,Transport,Transperth top-up,30.00,expense
2025-01-15,Income,Casual work cafe,320.00,income
2025-01-17,Food,Woolworths grocery,95.20,expense
2025-01-18,Utilities,Synergy electricity,145.60,expense
2025-01-20,Food,Sushi train dinner,38.00,expense
2025-01-21,Entertainment,Event Cinemas,24.50,expense
2025-01-22,Shopping,Cotton On clothing,59.99,expense
2025-01-25,Food,Uber Eats delivery,31.50,expense
2025-01-26,Health,Gym membership,49.00,expense
2025-01-28,Transport,Fuel Caltex,65.40,expense
2025-01-30,Utilities,Telstra mobile bill,55.00,expense
2025-01-31,Income,Payslip part time,1250.00,income
2025-02-01,Food,Coles grocery,91.80,expense
2025-02-03,Entertainment,Spotify premium,12.99,expense
2025-02-05,Shopping,Amazon order,44.99,expense
2025-02-07,Food,Cafe coffee,22.50,expense
2025-02-10,Health,GP visit gap payment,35.00,expense
2025-02-12,Food,Woolworths,78.60,expense
2025-02-14,Entertainment,Restaurant valentines,110.00,expense
2025-02-15,Income,Casual work cafe,290.00,income
2025-02-18,Transport,Fuel,72.10,expense
2025-02-20,Food,IGA,36.40,expense
2025-02-22,Utilities,Water bill,82.00,expense
2025-02-25,Shopping,JB Hi-Fi headphones,129.00,expense
2025-02-28,Income,Payslip part time,1250.00,income
2025-03-01,Food,Coles grocery,83.20,expense
2025-03-04,Transport,Fuel,69.30,expense
2025-03-06,Food,McDonalds,12.50,expense
2025-03-08,Entertainment,Disney plus subscription,13.99,expense
2025-03-10,Health,Gym membership,49.00,expense
2025-03-12,Food,Woolworths,88.90,expense
2025-03-15,Income,Payslip part time,1250.00,income
2025-03-17,Food,Uber Eats,27.80,expense
2025-03-19,Utilities,Synergy electricity,138.40,expense
2025-03-22,Shopping,Chemist,28.50,expense
2025-03-25,Entertainment,Concert tickets,89.00,expense
2025-03-28,Transport,Transperth top-up,30.00,expense
2025-03-31,Income,Casual work cafe,310.00,income
"""

finance_tips = """# Personal Finance Tips and Advice

## Budgeting Basics
The 50/30/20 rule is a popular budgeting guideline. Allocate 50 percent of after-tax income to needs such as rent, food, utilities, and transport. Allocate 30 percent to wants such as entertainment, dining out, and shopping. Allocate 20 percent to savings and debt repayment.

Zero-based budgeting means assigning every dollar of income a job before the month begins. Income minus all expenses, savings, and debt payments should equal zero.

Track every expense for at least one month before creating a budget. Most people underestimate their spending by 20 to 30 percent. Categories commonly underestimated include dining out, subscriptions, and small daily purchases like coffee.

## Managing Food Costs
Food is typically the most controllable major expense. Meal planning before grocery shopping can reduce food spending by 20 to 30 percent. Write a weekly menu, build your shopping list from it, and avoid shopping when hungry.

Cooking at home costs on average 5 to 7 times less than eating at a restaurant. If you spend 50 dollars per week on takeaway, switching to home cooking could save over 2000 dollars per year.

Grocery store home brand products are typically 20 to 40 percent cheaper than name brands. Applying this to staples like rice, pasta, tinned tomatoes, and dairy can meaningfully reduce your grocery bill.

## Subscriptions and Recurring Costs
Subscription creep is a common problem. Small monthly charges accumulate without notice. Audit all recurring payments every 3 months and cancel anything unused.

Streaming services: most households pay for 3 to 4 platforms averaging 60 to 80 dollars per month. Consider rotating services. Subscribe to one for a few months, cancel, and switch to another.

## Transport and Fuel
Fuel prices follow a weekly cycle in Australian cities. Prices are typically lowest on Tuesday or Wednesday. Using FuelWatch in WA can save 10 to 20 cents per litre.

Combining car trips reduces fuel consumption. Group errands into one outing. For distances under 5 kilometres, walking or cycling is free and improves health.

Public transport is significantly cheaper than driving when accounting for fuel, insurance, parking, and maintenance.

## Emergency Fund
An emergency fund is the foundation of financial security. Aim for 3 to 6 months of living expenses in an accessible savings account. Without an emergency fund, unexpected costs force you into high-interest debt.

Start small if needed. Even 500 dollars provides a buffer. Automate a transfer to savings on payday so it happens before discretionary spending.

High-yield savings accounts in Australia offer 4 to 5 percent interest. Move your emergency fund to UBank or ING rather than a basic transaction account earning near zero.

## Avoiding Debt Traps
Buy Now Pay Later services like Afterpay and Zip encourage spending beyond your means. The average BNPL user spends 20 to 30 percent more per purchase than with a debit card.

Credit card interest rates in Australia average 20 percent per year. Only use credit cards if you can commit to paying the full balance on the due date every month.

## Savings Goals
Name your savings account after your goal such as Japan Trip 2026 or New Laptop. Seeing the named goal makes transfers feel rewarding.

Automate savings transfers immediately after payday. Behavioural research consistently shows automating savings beats willpower alone.

The rule of 72: divide 72 by your annual interest rate to find how many years to double your money. At 4 percent, savings doubles in 18 years. Starting early matters enormously.

## Utility Bills
Switching electricity providers can save 200 to 500 dollars per year. Use the Energy Made Easy comparison tool in Australia.

Hot water accounts for 25 to 30 percent of home energy use. Shorter showers under 4 minutes reduce this cost significantly.

## Increasing Income
Side income from casual work, freelancing, or selling unused items accelerates financial goals. Even an extra 200 dollars per month invested over 10 years grows to over 36000 dollars at 5 percent return.

Review your pay against market rates annually using SEEK or LinkedIn Salary. Employees who negotiate receive raises averaging 7 to 10 percent above those who do not.

## Investing Basics
For long-term goals of 5 or more years, low-cost index funds historically outperform managed funds. ETFs like VAS and VGS have fees of 0.07 to 0.20 percent per year.

Superannuation is tax-advantaged. Employer contributions are taxed at 15 percent rather than your marginal rate. Voluntary contributions reduce taxable income while building retirement savings.

Dollar-cost averaging means investing a fixed amount regularly regardless of market conditions. It results in buying more units when prices are low and fewer when high.
"""

with open('data/expenses.csv', 'w') as f:
    f.write(csv_content)

with open('data/finance_tips.md', 'w') as f:
    f.write(finance_tips)

print('Data files created')

# ── STEP 4: CSV FUNCTIONS ────────────────────────────────────
def load_transactions(filepath):
    """
    Load a transactions CSV into a pandas DataFrame.
    Expected columns: date, category, description, amount, type
    Raises ValueError if required columns are missing.
    """
    required = {'date', 'category', 'description', 'amount', 'type'}
    df = pd.read_csv(filepath)
    df.columns = [c.strip().lower() for c in df.columns]
    missing = required - set(df.columns)
    if missing:
        raise ValueError('CSV is missing required columns: ' + str(missing))
    df['date']     = pd.to_datetime(df['date'], errors='coerce')
    df['amount']   = pd.to_numeric(df['amount'], errors='coerce').fillna(0)
    df['type']     = df['type'].str.strip().str.lower()
    df['category'] = df['category'].str.strip()
    return df


def analyse_spending(df):
    """
    Analyse a transactions DataFrame and return a formatted spending summary.
    Includes total income, total expenses, net balance, and category breakdown.
    """
    if df.empty:
        return 'No transaction data found.'

    expenses = df[df['type'] == 'expense']
    income   = df[df['type'] == 'income']

    total_income   = income['amount'].sum()
    total_expenses = expenses['amount'].sum()
    net            = total_income - total_expenses

    by_category = (
        expenses
        .groupby('category')['amount']
        .sum()
        .sort_values(ascending=False)
    )

    date_min = df['date'].min().strftime('%d %b %Y')
    date_max = df['date'].max().strftime('%d %b %Y')

    lines = [
        'Spending Summary ({} to {})'.format(date_min, date_max),
        '=' * 45,
        'Total Income:    ${:>10,.2f}'.format(total_income),
        'Total Expenses:  ${:>10,.2f}'.format(total_expenses),
        'Net Balance:     ${:>10,.2f} {}'.format(net, '(surplus)' if net >= 0 else '(deficit)'),
        '',
        'Spending by Category:',
        '-' * 35,
    ]

    for cat, amt in by_category.items():
        pct = (amt / total_expenses * 100) if total_expenses > 0 else 0
        bar = '#' * int(pct / 5)
        lines.append('  {:<15} ${:>8,.2f}  ({:4.1f}%) {}'.format(cat, amt, pct, bar))

    if not by_category.empty:
        lines.append('')
        lines.append('Top spend: {} (${:,.2f})'.format(
            by_category.index[0], by_category.iloc[0]))

    return '\n'.join(lines)


print('CSV functions defined')

# ── STEP 5: RAG FUNCTIONS ────────────────────────────────────
def build_rag_index(source_file, index_path='data/finance_index'):
    """
    Build a RAG index from a markdown or text file.
    Chunks the text, generates embeddings, and saves a .npz index file.
    """
    with open(source_file, 'r', encoding='utf-8') as f:
        text = f.read()
    print('Loaded document:', len(text), 'characters')
    chunks = chunk_text(text, chunk_size=300)
    print('Created', len(chunks), 'chunks')
    print('Generating embeddings, this may take a moment...')
    vectors = get_embeddings(chunks)
    sources = [source_file] * len(chunks)
    save_index_with_sources(vectors, chunks, sources, index_path)
    print('Index saved to', index_path + '.npz')
    return index_path


def rag_answer(question, index_path, k=3):
    """
    Answer a finance question using the RAG index.
    Retrieves top-k relevant chunks and passes them as context to the LLM.
    """
    results = get_top_k(question, index_path, k=k)
    if not results:
        return 'I could not find relevant information in the knowledge base for that question.'
    context = '\n\n---\n\n'.join(chunk for chunk, *_ in results)
    prompt = (
        'You are a helpful personal finance advisor.\n'
        'Answer the following question using ONLY the provided context.\n'
        'Be practical and specific. If the context is insufficient, say so.\n\n'
        'Context:\n' + context + '\n\n'
        'Question: ' + question + '\n\nAnswer:'
    )
    return get_response(prompt)


INDEX_PATH = 'data/finance_index.npz' # Corrected to include .npz extension
build_rag_index('data/finance_tips.md', INDEX_PATH.replace('.npz', '')) # Pass path without extension to save_index_with_sources
print('RAG index ready')

# ── STEP 6: SAVINGS GOAL TOOL ────────────────────────────────
def savings_goal_calculator(target, current, monthly):
    """
    Calculate how many months it takes to reach a savings goal.

    Args:
        target  : Total savings goal in dollars
        current : Amount already saved
        monthly : Amount saved per month

    Returns:
        Human-readable string with the savings timeline.
    """
    target  = float(target)
    current = float(current)
    monthly = float(monthly)

    if target <= 0:
        return 'Error: Target amount must be greater than zero.'
    if monthly <= 0:
        return 'Error: Monthly savings amount must be greater than zero.'
    if current < 0:
        return 'Error: Current savings cannot be negative.'
    if current >= target:
        return 'You have already reached your goal of ${:,.2f}!'.format(target)

    remaining = target - current
    months    = math.ceil(remaining / monthly)
    years     = round(months / 12, 1)

    lines = [
        'Savings Goal Calculator',
        '-' * 35,
        '  Goal:             ${:>10,.2f}'.format(target),
        '  Already saved:    ${:>10,.2f}'.format(current),
        '  Still needed:     ${:>10,.2f}'.format(remaining),
        '  Monthly saving:   ${:>10,.2f}'.format(monthly),
        '-' * 35,
        '  Time to goal: {} months ({} years)'.format(months, years),
    ]

    if months <= 6:
        lines.append('  Almost there, keep it up!')
    elif months <= 12:
        lines.append('  You will reach it within a year, great progress!')
    elif months <= 24:
        lines.append('  Solid plan. See if you can increase your monthly savings.')
    else:
        lines.append('  Tip: Increasing monthly savings by 10% would cut this time significantly.')

    return '\n'.join(lines)


register_tool(
    name='savings_goal_calculator',
    description=(
        'Calculates how many months it will take to reach a savings goal. '
        'Input: target amount (float), current savings (float), monthly saving amount (float). '
        'Output: formatted string with timeline. '
        'Example: savings_goal_calculator(2000, 400, 150)'
    ),
    function=savings_goal_calculator
)
print('Savings tool registered')

# ── STEP 7: PENNY CHATBOT ────────────────────────────────────
PENNY_SYSTEM_PROMPT = (
    'You are Penny, a friendly and encouraging personal finance coach for Australian '
    'university students and young workers. Your personality:\n'
    '- Warm, supportive, and non-judgmental\n'
    '- Practical and specific, give actionable advice not vintage platitudes\n'
    '- Uses Australian context such as AUD, Afterpay, Coles, Woolworths, Synergy\n'
    '- Concise, 2 to 4 paragraphs per response\n'
    '- Honest when you do not know something\n'
    'Focus areas: budgeting, tracking spending, reducing costs, savings habits, debt, goals.\n'
    'You are NOT a licensed financial advisor. For major investment decisions '
    'always recommend consulting a professional.'
)


def penny_chat(user_message, history, spending_context=''):
    """
    Send a message to Penny and get a personalised response.\n
    Args:
        user_message     : The user question or message
        history          : Conversation history list of user and bot pairs
        spending_context : Optional spending summary to include as context\n
    Returns:
        Penny response as a string
    """
    if spending_context:
        full_prompt = (
            'The user has shared their transaction data:\n'
            + spending_context
            + '\n\nUser says: '
            + user_message
        )
    else:
        full_prompt = user_message
    return get_response(full_prompt, system=PENNY_SYSTEM_PROMPT)


print('Penny chatbot defined')

# ── STEP 8: GRADIO UI ────────────────────────────────────────
app_state = {'df': None, 'summary': ''}


def handle_csv_upload(file):
    if file is None:
        return 'Please upload a CSV file first.'
    try:
        df = load_transactions(file.name)
        summary = analyse_spending(df)
        app_state['df']      = df
        app_state['summary'] = summary
        return summary
    except ValueError as e:
        return 'Error reading CSV: ' + str(e)
    except Exception as e:
        return 'Unexpected error: ' + str(e)


def handle_penny_chat(user_message, history):
    if not user_message.strip():
        return history, ''
    context  = app_state.get('summary', '')
    response = penny_chat(user_message, history, spending_context=context)
    history  = history + [(user_message, response)]
    return history, ''


def handle_rag_question(question):
    if not question.strip():
        return 'Please enter a question.'
    return rag_answer(question, INDEX_PATH)


def handle_savings_goal(target, current, monthly):
    try:
        return savings_goal_calculator(float(target), float(current), float(monthly))
    except (ValueError, TypeError):
        return 'Please enter valid numbers for all fields.'


with gr.Blocks(title='Smart Finance Assistant', theme=gr.themes.Soft()) as app:

    gr.Markdown('# Smart Finance Assistant\n*Your personal finance coach powered by AI*\n---')

    with gr.Tabs():

        with gr.TabItem('Spending Analysis'):
            gr.Markdown(
                '### Upload your transaction CSV\n'
                'File must have columns: date, category, description, amount, type\n'
                'Use data/expenses.csv to try it out.'
            )
            with gr.Row():
                csv_input  = gr.File(label='Upload CSV', file_types=['.csv'])
                csv_output = gr.Textbox(
                    label='Spending Summary',
                    lines=20,
                    placeholder='Upload a CSV to see your spending breakdown'
                )
            gr.Button('Analyse Spending', variant='primary').click(
                fn=handle_csv_upload,
                inputs=[csv_input],
                outputs=[csv_output]
            )

        with gr.TabItem('Chat with Penny'):
            gr.Markdown(
                '### Ask Penny anything about your finances\n'
                'Tip: Upload your CSV first so Penny can give personalised advice.'
            )
            chatbot = gr.Chatbot(
                value=[(None, 'Hi! I am Penny, your friendly finance coach. '
                              'How can I help today? Upload your transactions for personalised advice!')],
                label='Penny',
                height=400
            )
            with gr.Row():
                msg_box  = gr.Textbox(
                    placeholder='Ask Penny something, e.g. How can I cut my food spending?',
                    label='Your message',
                    scale=5
                )
                send_btn = gr.Button('Send', variant='primary', scale=1)
            send_btn.click(handle_penny_chat, [msg_box, chatbot], [chatbot, msg_box])
            msg_box.submit(handle_penny_chat, [msg_box, chatbot], [chatbot, msg_box])
            gr.Examples(
                examples=[
                    ['I keep spending too much on food. Any tips.'],
                    ['How do I start building an emergency fund.'],
                    ['Is Afterpay a good idea.'],
                    ['What is the 50/30/20 rule.'],
                ],
                inputs=[msg_box]
            )

        with gr.TabItem('Finance Q and A'):
            gr.Markdown(
                '### Ask finance questions answered from a curated knowledge base\n'
                'Answers are grounded in real finance advice.'
            )
            rag_input  = gr.Textbox(
                label='Your finance question',
                placeholder='e.g. How does the 50/30/20 rule work?',
                lines=2
            )
            rag_output = gr.Textbox(label='Answer', lines=10)
            gr.Button('Find Answer', variant='primary').click(
                fn=handle_rag_question,
                inputs=[rag_input],
                outputs=[rag_output]
            )
            gr.Examples(
                examples=[
                    ['How can I reduce my electricity bill.'],
                    ['What is dollar-cost averaging.'],
                    ['Why is an emergency fund important.'],
                    ['Should I use Afterpay.'],
                ],
                inputs=[rag_input]
            )

        with gr.TabItem('Savings Goal'):
            gr.Markdown('### How long will it take to reach your savings goal?')
            with gr.Row():
                with gr.Column():
                    goal_target  = gr.Number(label='Goal Amount in dollars',         value=2000)
                    goal_current = gr.Number(label='Current Savings in dollars',      value=0)
                    goal_monthly = gr.Number(label='Monthly Contribution in dollars', value=100)
                    calc_btn     = gr.Button('Calculate', variant='primary')
                goal_output = gr.Textbox(label='Your Savings Timeline', lines=12)
            calc_btn.click(
                fn=handle_savings_goal,
                inputs=[goal_target, goal_current, goal_monthly],
                outputs=[goal_output]
            )
            gr.Examples(
                examples=[[2000, 400, 150], [5000, 0, 200], [10000, 2000, 300]],
                inputs=[goal_target, goal_current, goal_monthly],
                label='Example Goals'
            )

print('Gradio app built')

# ── STEP 9: TESTS ────────────────────────────────────────────
print()
print('Running all tests...')
print('=' * 50)

# --- Test Group 1: CSV ---
df_t = load_transactions('data/expenses.csv')
assert len(df_t) > 0,            'T1.1 failed: CSV should load rows'
assert 'amount' in df_t.columns, 'T1.1 failed: missing amount column'
assert 'type'   in df_t.columns, 'T1.1 failed: missing type column'
print('T1.1 PASS - CSV loads correctly')

s = analyse_spending(df_t)
assert 'Total Income'   in s, 'T1.2 failed: missing Total Income'
assert 'Total Expenses' in s, 'T1.2 failed: missing Total Expenses'
assert 'Net Balance'    in s, 'T1.2 failed: missing Net Balance'
assert 'Food'           in s, 'T1.2 failed: missing Food category'
print('T1.2 PASS - Summary contains expected sections')

empty_df = pd.DataFrame(columns=['date', 'category', 'description', 'amount', 'type'])
assert 'No transaction data' in analyse_spending(empty_df), 'T1.3 failed: empty DF not handled'
print('T1.3 PASS - Empty DataFrame handled gracefully')

known = pd.DataFrame([
    {'date': '2025-01-01', 'category': 'Food',      'description': 'Coles', 'amount': 100.00, 'type': 'expense'},
    {'date': '2025-01-02', 'category': 'Transport',  'description': 'Fuel',  'amount': 50.00,  'type': 'expense'},
    {'date': '2025-01-03', 'category': 'Income',     'description': 'Pay',   'amount': 1000.00,'type': 'income'},
])
known['date'] = pd.to_datetime(known['date'])
ks = analyse_spending(known)
assert '1,000.00' in ks, 'T1.4 failed: income total wrong'
assert '150.00'   in ks, 'T1.4 failed: expense total wrong'
assert '850.00'   in ks, 'T1.4 failed: net balance wrong'
print('T1.4 PASS - Totals calculated correctly')

bad = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
bad.write('col1,col2\\n1,2\\n')
bad.close()
try:
    load_transactions(bad.name)
    assert False, 'T1.5 failed: should have raised ValueError'
except ValueError:
    pass
finally:
    os.unlink(bad.name)
print('T1.5 PASS - Invalid CSV raises ValueError')

# --- Test Group 2: Savings Calculator ---
r = savings_goal_calculator(2000, 400, 150)
assert '11 months' in r, 'T2.1 failed: expected 11 months, got: ' + r
print('T2.1 PASS - Standard calculation correct')

r = savings_goal_calculator(1000, 1500, 200)
assert 'already reached' in r.lower(), 'T2.2 failed: goal already reached not detected'
print('T2.2 PASS - Already reached goal detected')

r = savings_goal_calculator(1000, 0, 0)
assert 'Error' in r, 'T2.3 failed: zero monthly should return error'
print('T2.3 PASS - Zero monthly savings returns error')

r = savings_goal_calculator(-500, 0, 100)
assert 'Error' in r, 'T2.4 failed: negative target should return error'
print('T2.4 PASS - Negative target returns error')

r = savings_goal_calculator(600, 0, 100)
assert '6 months' in r, 'T2.5 failed: expected 6 months, got: ' + r
print('T2.5 PASS - Exact divisor gives correct result')

r = savings_goal_calculator(101, 0, 100)
assert '2 months' in r, 'T2.6 failed: expected 2 months (ceiling), got: ' + r
print('T2.6 PASS - Ceiling applied correctly')

r = savings_goal_calculator(50000, 5000, 500)
assert 'months' in r, 'T2.7 failed: should return month count'
assert 'years'  in r, 'T2.7 failed: should return year count'
print('T2.7 PASS - Large goal returns months and years')

# --- Test Group 3: RAG ---
assert os.path.exists(INDEX_PATH), 'T3.1 failed: index file missing'
print('T3.1 PASS - RAG index file exists')

answer = rag_answer('What is an emergency fund?', INDEX_PATH)
assert isinstance(answer, str) and len(answer) > 50, 'T3.2 failed: RAG answer too short'
print('T3.2 PASS - RAG returns non-empty answer')

answer = rag_answer('How does the 50/30/20 rule work?', INDEX_PATH)
relevant_terms = ['50', '30', '20', 'budget', 'income', 'needs', 'wants']
assert any(t.lower() in answer.lower() for t in relevant_terms), 'T3.3 failed: answer not on topic'
print('T3.3 PASS - RAG retrieves topically relevant content')

# --- Test Group 4: Penny ---
response = penny_chat('What is a budget?', history=[])
assert isinstance(response, str) and len(response) > 20, 'T4.1 failed: response too short'
print('T4.1 PASS - Penny returns non-empty response')

fake_context = 'Total Income: $1,000 | Total Expenses: $800 | Top spend: Food $250'
response = penny_chat('Am I spending too much on food?', history=[], spending_context=fake_context)
assert isinstance(response, str) and len(response) > 20, 'T4.2 failed: response with context too short'
print('T4.2 PASS - Penny handles spending context')

print()
print('All tests passed!')

# ── STEP 10: LAUNCH ──────────────────────────────────────────
print()
print('Launching Gradio app...')
app.launch(share=True)


Packages imported and configuration set (Timeout increased to 120s)
Data files created
CSV functions defined
Loaded document: 4843 characters
Created 3 chunks
Generating embeddings, this may take a moment...
